# 01. Dataset Exploration & Problem Formulation
**Project:** Academic Sports Machine Learning Project  
**Domain:** Cricket Player Performance Analytics  
**Course Context:** Module 1 (ML Lifecycle & Data Understanding)  

---

## 1. Executive Summary & Objective
This project analyzes individual player match records across international and league cricket matches.
The objective of this notebook is to:
1. Load and inspect the sports dataset (`player_performance_dataset_10000.csv`).
2. Audit dimensions, feature types, missing values, and potential duplicates.
3. Identify candidate target variables and establish the formal **Supervised Machine Learning problem**.
4. Prevent data leakage by isolating identifiers and evaluating candidate targets.


In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Paths
DATA_PATH = Path("../data/raw/player_performance_dataset_10000.csv")
TABLES_DIR = Path("../outputs/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset Loaded Successfully! Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(5)

Dataset Loaded Successfully! Shape: 10000 rows x 31 columns


,Player_ID,Player_Name,Age,Role,Team,Match_ID,Match_Date,Match_Format,Opponent,Venue,...,Runs_Conceded,Economy,Catches,Stumpings,Recent_Form,Previous_Performance,Fitness_Score,Consistency_Score,Pressure_Score,Performance_Score
0,P00001,Player_00001,35,All-Rounder,Team_A,M000001,2022-07-10,Test,Team_C,Stadium_28,...,75.9,9.37,0,0,55.86,46.77,66.68,57.62,67.08,32.78
1,P00002,Player_00002,21,Bowler,Team_H,M000002,2025-09-15,Test,Team_D,Stadium_23,...,11.6,4.83,3,0,39.64,53.46,75.12,82.59,61.85,38.11
2,P00003,Player_00003,25,All-Rounder,Team_C,M000003,2024-06-11,ODI,Team_I,Stadium_07,...,0.0,0.00,4,0,63.26,72.61,68.69,58.28,51.79,59.42
3,P00004,Player_00004,21,All-Rounder,Team_H,M000004,2024-08-30,T20,Team_B,Stadium_30,...,28.1,6.11,4,0,58.07,70.24,82.90,90.99,70.03,63.87
4,P00005,Player_00005,28,Batsman,Team_F,M000005,2023-12-29,T20,Team_B,Stadium_30,...,28.8,5.54,2,0,51.07,64.58,91.17,63.26,69.95,51.85


In [ ]:
# Column types and null counts
info_df = pd.DataFrame({
    'Column': df.columns,
    'Dtype': df.dtypes.astype(str),
    'Null_Count': df.isnull().sum(),
    'Null_Pct': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique_Values': df.nunique()
}).reset_index(drop=True)

print("--- Column Inventory ---")
print(info_df.to_string())
info_df.to_csv(TABLES_DIR / "dataset_inventory.csv", index=False)

--- Column Inventory ---
                  Column    Dtype  Null_Count  Null_Pct  Unique_Values
0              Player_ID      str           0       0.0          10000
1            Player_Name      str           0       0.0          10000
2                    Age    int64           0       0.0             23
3                   Role      str           0       0.0              4
4                   Team      str           0       0.0             10
5               Match_ID      str           0       0.0          10000
6             Match_Date      str           0       0.0           1669
7           Match_Format      str           0       0.0              3
8               Opponent      str           0       0.0             10
9                  Venue      str           0       0.0             30
10             Home_Away      str           0       0.0              2
11      Experience_Years    int64           0       0.0             25
12        Matches_Played    int64           0       

## 3. Duplicate and Identifier Check
We check for exact row duplicates and investigate player/match identifier uniqueness.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Total Exact Duplicate Rows: {duplicates}")

# Identifier Cardinality
print(f"Unique Player IDs: {df['Player_ID'].nunique()}")
print(f"Unique Match IDs:  {df['Match_ID'].nunique()}")
assert duplicates == 0, "Warning: Unexpected duplicates found!"
print("Data integrity check passed: 0 missing values, 0 duplicate rows.")

Total Exact Duplicate Rows: 0


Unique Player IDs: 10000
Unique Match IDs:  10000
Data integrity check passed: 0 missing values, 0 duplicate rows.


## 4. Summary Statistics for Numerical Attributes
Understanding the central tendency, spread, and physical limits of cricket performance metrics.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
num_summary = df[num_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
num_summary['IQR'] = num_summary['75%'] - num_summary['25%']
print(num_summary.round(2).to_string())
num_summary.round(2).to_csv(TABLES_DIR / "numerical_summary.csv")

                        mean     std    min    25%     50%     75%      max     IQR
Age                    29.10    6.65  18.00  23.00   29.00   35.00    40.00   12.00
Experience_Years        9.36    6.72   0.00   3.00    9.00   15.00    24.00   12.00
Matches_Played        100.13   57.65   1.00  50.00  100.00  150.00   200.00  100.00
Runs                   94.95   37.99   0.00  69.00   95.00  121.00   230.00   52.00
Balls_Faced           114.79   47.42   1.00  81.00  115.00  148.00   291.00   67.00
Fours                   9.47    4.27   0.00   7.00    9.00   12.00    25.00    5.00
Sixes                   4.80    2.67   0.00   3.00    5.00    7.00    14.00    4.00
Strike_Rate            96.19  116.63   0.00  67.04   82.39  105.60  3000.00   38.56
Batting_Average        52.72   14.64   5.00  42.86   52.77   62.60   100.00   19.74
Wickets                 2.09    1.55   0.00   1.00    2.00    3.00     9.00    2.00
Overs_Bowled            4.52    2.20   0.00   3.00    4.50    6.00    14.60 

## 5. Categorical Feature Distributions
Analyzing the distinct categories in Role, Match Format, Team, and Venue.

In [ ]:
cat_cols = ['Role', 'Match_Format', 'Home_Away', 'Team', 'Opponent']
for c in cat_cols:
    print(f"--- Distribution for {c} ---")
    print(df[c].value_counts().to_string())
    print()

--- Distribution for Role ---
Role
Batsman          3834
Bowler           2837
All-Rounder      2162
Wicket-Keeper    1167

--- Distribution for Match_Format ---
Match_Format
T20     5503
ODI     3520
Test     977

--- Distribution for Home_Away ---
Home_Away
Away    5028
Home    4972

--- Distribution for Team ---
Team
Team_B    1037
Team_G    1033
Team_H    1028
Team_I    1009
Team_J    1006
Team_E    1000
Team_C     991
Team_D     989
Team_F     977
Team_A     930

--- Distribution for Opponent ---
Opponent
Team_G    1034
Team_J    1029
Team_A    1028
Team_C    1024
Team_H     995
Team_I     989
Team_B     984
Team_F     981
Team_E     979
Team_D     957



## 6. Target Variable Analysis & Problem Formulation
The dataset contains a composite metric: `Performance_Score` (range 4.35 to 100.0, mean ~50.45).

### Academic Problem Formulation: Elite Player Classification
While `Performance_Score` can be modeled as a continuous regression target, real-world sports decision-makers (coaches, franchise selectors, fantasy leagues) must make discrete binary decisions:
- **Which players delivered an elite, match-winning performance?** (e.g. Player of the Match candidate / Dream Team selection).
- A threshold of `Performance_Score >= 60.0` isolates the top ~19% of match performances.
- This creates an academically realistic **class-imbalanced problem (~1:4.25 ratio)**, necessitating class-weighting, PR-AUC, and resampling in **Notebook 06 (`06_Imbalance_Handling.ipynb`)**.


In [ ]:
# Define binary target
THRESHOLD = 60.0
df['High_Performer'] = (df['Performance_Score'] >= THRESHOLD).astype(int)

class_dist = df['High_Performer'].value_counts()
class_pct = df['High_Performer'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Class': ['Standard Performer (0)', 'High Performer (1)'],
    'Count': class_dist.values,
    'Percentage': class_pct.values.round(2)
})
print("Target Variable Distribution:")
print(target_summary.to_string(index=False))
target_summary.to_csv(TABLES_DIR / "target_distribution.csv", index=False)

Target Variable Distribution:
                 Class  Count  Percentage
Standard Performer (0)   8096       80.96
    High Performer (1)   1904       19.04


## 7. Data Leakage Audit
To maintain strict ML integrity:
1. `Player_ID`, `Player_Name`, and `Match_ID` are identifiers and must NOT be used as predictive features.
2. `Performance_Score` was used to derive `High_Performer`. It **must be dropped from features $X$** to prevent target leakage!


In [ ]:
leakage_cols = ['Player_ID', 'Player_Name', 'Match_ID', 'Performance_Score']
print(f"Features flagged for removal prior to modeling: {leakage_cols}")

Features flagged for removal prior to modeling: ['Player_ID', 'Player_Name', 'Match_ID', 'Performance_Score']


## 8. Academic Viva Questions & Defense Notes

### Q1: Why formulate this as binary classification rather than regression?
**Defense:** In competitive sports, selection decisions are binary (selected vs not selected, elite award vs standard match performance). Formulating this as identifying high-impact players (`Performance_Score >= 60.0`) mirrors the real-world operational problem of talent identification. Moreover, it introduces natural class imbalance (~19% positive class), allowing rigorous study of precision-recall dynamics, cost-sensitive learning, and resampling techniques required by the curriculum.

### Q2: How did you verify there is no data leakage?
**Defense:** The continuous `Performance_Score` is strictly excluded from the feature matrix $X$. Additionally, non-generalizable identifiers (`Player_ID`, `Match_ID`, `Player_Name`) are dropped.
